# DATA37711 Homework 2 Question 1

In [9]:
import copy
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import TensorDataset, DataLoader

from scipy.stats import norm
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import brier_score_loss, accuracy_score

## (c)

In [10]:
# 1. Reproducibility and data loading
np.random.seed(42)
torch.manual_seed(42)

df = pd.read_parquet("taxi_trips.parquet")

X = df.drop(columns="trip_duration").to_numpy(dtype=np.float32)
y = df["trip_duration"].to_numpy(dtype=np.float32)

assert X.shape[1] == 7
assert np.isfinite(X).all() and np.isfinite(y).all()

# 2. Split data: 70% training / 20% validation / 10% test
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.30,
    random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=1 / 3,
    random_state=42
)

print(f"Training samples:   {len(y_train)}")
print(f"Validation samples: {len(y_val)}")
print(f"Test samples:       {len(y_test)}")


# 3. Standardize the target using training statistics only

# The input features have already been standardized.
y_mean = float(y_train.mean())
y_std = max(float(y_train.std()), 1e-6)

z_train = (y_train - y_mean) / y_std
z_val = (y_val - y_mean) / y_std

train_data = TensorDataset(
    torch.from_numpy(X_train),
    torch.from_numpy(z_train)
)

val_data = TensorDataset(
    torch.from_numpy(X_val),
    torch.from_numpy(z_val)
)

train_loader = DataLoader(
    train_data,
    batch_size=256,
    shuffle=True
)

val_loader = DataLoader(
    val_data,
    batch_size=1024,
    shuffle=False
)


# 4. Define the ReLU MLP
class GaussianMLP(nn.Module):
    def __init__(self):
        super().__init__()

        self.layers = nn.Sequential(
            nn.Linear(7, 64),
            nn.ReLU(),
            nn.Linear(64, 64),
            nn.ReLU(),
            nn.Linear(64, 2)
        )

    def forward(self, x):
        output = self.layers(x)

        # Identity link for the mean
        mu = output[:, 0]

        # Softplus link for the standard deviation
        sigma = F.softplus(output[:, 1]) + 1e-4

        return mu, sigma


# 5. Gaussian negative log-likelihood
def gaussian_nll(y, mu, sigma):
    return (
        0.5 * np.log(2 * np.pi)
        + torch.log(sigma)
        + (y - mu) ** 2 / (2 * sigma ** 2)
    ).mean()


model = GaussianMLP()

optimizer = torch.optim.SGD(
    model.parameters(),
    lr=0.01
)


# 6. Train and select the best epoch using validation NLL
max_epochs = 300
patience = 20

best_val_loss = float("inf")
best_state = None
best_epoch = None
wait = 0

for epoch in range(1, max_epochs + 1):

    # Training
    model.train()
    train_total = 0.0

    for xb, yb in train_loader:
        optimizer.zero_grad()

        mu, sigma = model(xb)
        loss = gaussian_nll(yb, mu, sigma)

        if not torch.isfinite(loss).item():
            raise RuntimeError(
                "Nonfinite training loss. Try a smaller learning rate."
            )

        loss.backward()

        nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=5.0
        )

        optimizer.step()

        train_total += loss.item() * len(yb)

    train_loss = train_total / len(train_data)

    # Validation
    model.eval()
    val_total = 0.0

    with torch.no_grad():
        for xb, yb in val_loader:
            mu, sigma = model(xb)
            loss = gaussian_nll(yb, mu, sigma)

            val_total += loss.item() * len(yb)

    val_loss = val_total / len(val_data)

    if not np.isfinite(val_loss):
        raise RuntimeError(
            "Nonfinite validation loss. Try a smaller learning rate."
        )

    # Report NLL in the original duration units
    print(
        f"Epoch {epoch:03d} | "
        f"Train NLL: {train_loss + np.log(y_std):.4f} | "
        f"Validation NLL: {val_loss + np.log(y_std):.4f}"
    )

    # Save the parameters with the lowest validation NLL
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_state = copy.deepcopy(model.state_dict())
        best_epoch = epoch
        wait = 0
    else:
        wait += 1

    if wait >= patience:
        print(f"\nEarly stopping at epoch {epoch}.")
        break


# Restore the best MLP, rather than the final epoch
model.load_state_dict(best_state)
model.eval()

mlp_val_nll = best_val_loss + np.log(y_std)


# 7. Fit the linear Gaussian baseline
baseline = LinearRegression()
baseline.fit(X_train, y_train)

train_residuals = y_train - baseline.predict(X_train)

baseline_sigma = max(
    float(np.sqrt(np.mean(train_residuals ** 2))),
    1e-6
)

baseline_mu_val = baseline.predict(X_val)

baseline_val_nll = np.mean(
    0.5 * np.log(2 * np.pi)
    + np.log(baseline_sigma)
    + (y_val - baseline_mu_val) ** 2 / (2 * baseline_sigma ** 2)
)


# 8. Print the selected MLP and baseline comparison
comparison = pd.DataFrame({
    "Model": [
        "Linear Gaussian baseline",
        "Best Gaussian MLP"
    ],
    "Validation NLL": [
        baseline_val_nll,
        mlp_val_nll
    ]
})

print(f"\nSelected MLP epoch: {best_epoch}")
print(comparison.round(4).to_string(index=False))

best_model_name = (
    "Gaussian MLP"
    if mlp_val_nll <= baseline_val_nll
    else "Linear Gaussian baseline"
)

print(f"\nLowest validation NLL: {best_model_name}")
print("The selected MLP parameters are loaded in `model`.")

Training samples:   366487
Validation samples: 104711
Test samples:       52356
Epoch 001 | Train NLL: 3.2235 | Validation NLL: 3.0262
Epoch 002 | Train NLL: 3.2094 | Validation NLL: 2.8513
Epoch 003 | Train NLL: 2.7873 | Validation NLL: 2.8109
Epoch 004 | Train NLL: 2.7551 | Validation NLL: 2.7963
Epoch 005 | Train NLL: 2.7359 | Validation NLL: 2.7790
Epoch 006 | Train NLL: 2.7231 | Validation NLL: 2.7721
Epoch 007 | Train NLL: 2.7124 | Validation NLL: 2.7814
Epoch 008 | Train NLL: 2.7050 | Validation NLL: 2.7463
Epoch 009 | Train NLL: 2.6989 | Validation NLL: 2.7520
Epoch 010 | Train NLL: 2.6936 | Validation NLL: 2.7353
Epoch 011 | Train NLL: 2.6882 | Validation NLL: 2.7350
Epoch 012 | Train NLL: 2.6862 | Validation NLL: 2.7369
Epoch 013 | Train NLL: 2.6790 | Validation NLL: 2.7474
Epoch 014 | Train NLL: 2.6776 | Validation NLL: 2.7218
Epoch 015 | Train NLL: 2.6727 | Validation NLL: 2.7250
Epoch 016 | Train NLL: 2.6708 | Validation NLL: 2.7418
Epoch 017 | Train NLL: 2.6687 | Validati

In [13]:
print(f"Total epochs trained: {epoch}")
print(f"Best epoch: {best_epoch}")
print(f"Best MLP validation NLL: {mlp_val_nll:.4f}")
print(f"Baseline validation NLL: {baseline_val_nll:.4f}")


Total epochs trained: 74
Best epoch: 54
Best MLP validation NLL: 2.6835
Baseline validation NLL: 3.4390


## (d)

In [11]:
# Select one trip
x = torch.from_numpy(X_test[0:1])

model.eval()

with torch.no_grad():
    mu_z, sigma_z = model(x)

# Convert predictions back to minutes
predicted_mu = y_mean + y_std * mu_z.item()
predicted_sigma = y_std * sigma_z.item()

# P(Y < 45 | X = x)
probability = norm.cdf(
    (45 - predicted_mu) / predicted_sigma
)

print(f"Predicted mean: {predicted_mu:.4f} minutes")
print(f"Predicted standard deviation: {predicted_sigma:.4f} minutes")
print(f"P(trip duration < 45 minutes): {probability:.6f}")

Predicted mean: 5.0898 minutes
Predicted standard deviation: 3.9184 minutes
P(trip duration < 45 minutes): 1.000000


## (e)

In [12]:
# Predict in batches for the entire test set
test_loader = DataLoader(
    TensorDataset(torch.from_numpy(X_test)),
    batch_size=1024,
    shuffle=False
)

mu_list = []
sigma_list = []

model.eval()

with torch.no_grad():
    for (xb,) in test_loader:
        mu_z, sigma_z = model(xb)

        mu_list.append(mu_z.numpy())
        sigma_list.append(sigma_z.numpy())

# Convert predictions back to minutes
mu_test = y_mean + y_std * np.concatenate(mu_list)
sigma_test = y_std * np.concatenate(sigma_list)

# Predicted probabilities: P(Y < 45 | X)
probabilities = norm.cdf(
    (45 - mu_test) / sigma_test
)

# True labels: 1 if duration < 45, otherwise 0
true_labels = (y_test < 45).astype(int)

# Classify using a probability threshold of 0.5
predicted_labels = (probabilities >= 0.5).astype(int)

# Evaluate
brier = brier_score_loss(true_labels, probabilities)
accuracy = accuracy_score(true_labels, predicted_labels)

print(f"Test Brier score: {brier:.6f}")
print(f"Test classification accuracy: {accuracy:.4f}")

Test Brier score: 0.002997
Test classification accuracy: 0.9964
